# 참조 얼굴 생성

GPT-Image 로 조합 3 참조 얼굴 풀을 만든다. GPU 는 쓰지 않는다.
저장 위치는 MyDrive/saloncut_data/ref_faces/

## 이력
- ref-01~32  최초 32장
- ref-33~38  8/25 추가. 팀 피드백 "얼굴 변형이 적다" 대응.
             기존 매력형이 attractive and photogenic 수준이라
             한국인 평균의 변주에 머물렀다. 강도 표현을 올렸다.

In [ ]:
!pip install -q openai

from getpass import getpass
from pathlib import Path

from google.colab import drive
from openai import OpenAI

drive.mount("/content/drive")
REF_DIR = Path("/content/drive/MyDrive/saloncut_data/ref_faces")
print("기존 파일 수:", len(list(REF_DIR.glob("ref-*.png"))))

client = OpenAI(api_key=getpass("OpenAI API key: "))
print("준비 완료")

In [ ]:
SUFFIX = (
    "completely bald with no hair on the head, eyebrows present, "
    "facing directly forward, eyes open looking at camera, "
    "neutral relaxed expression, head and shoulders only, plain black top, "
    "solid light gray background, even soft studio lighting, minimal shadows, "
    "no makeup, no accessories, no glasses, photorealistic portrait photograph, "
    "sharp focus on the face"
)

BEAUTY = (
    "a Korean woman in her early 20s, elite high fashion runway model, "
    "extremely small head, unusually large eyes for the face width, "
    "very short philtrum, narrow tapered jaw, high smooth forehead, "
    "long slender neck, flawless poreless skin"
)

FACES = {
    "ref-33": "very slim oval face, extremely large wide-set double-lidded eyes taking up an unusually wide portion of the face, clean sharp jawline tapering to a small chin, high straight thin nose bridge, small delicate lips, very short philtrum",
    "ref-34": "narrow V-line face with high prominent cheekbones, deeply set very large eyes with thick long lashes, sharply sculpted high nose bridge with narrow tip, full wide lips, strong angular bone structure",
    "ref-35": "small compact face with soft rounded jaw and tiny pointed chin, very large round clear eyes set low on the face, small refined button nose, high wide forehead, pale luminous skin, youthful proportions",
    "ref-36": "slim angular face, long narrow upturned almond eyes with sharply lifted outer corners, wide-set eyes, thin high nose bridge, defined lips with sharp cupid bow, prominent cheekbones, feline sophisticated impression",
    "ref-37": "exceptionally small face with very narrow pointed chin, enormously large round eyes with long lashes occupying nearly a third of the face height, tiny straight nose, very short philtrum, small full lips, doll-like exaggerated proportions",
    "ref-38": "elongated narrow face with strong defined jawline, long narrow sharp eyes with cool downturned gaze, very high thin nose bridge, thin precisely defined lips, hollow cheeks, aloof editorial impression",
}

In [ ]:
import base64
import io
import time

from PIL import Image
import matplotlib.pyplot as plt

results = {}
for ref_id, detail in FACES.items():
    prompt = f"{BEAUTY}, {detail}, {SUFFIX}"
    t = time.time()

    res = client.images.generate(
        model="gpt-image-2", prompt=prompt, size="1024x1024", n=1
    )
    img = Image.open(io.BytesIO(base64.b64decode(res.data[0].b64_json))).convert("RGB")

    img.save(REF_DIR / f"{ref_id}.png")
    results[ref_id] = img
    print(f"{ref_id}  {time.time() - t:.0f}s")

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, (ref_id, img) in zip(axes.flat, results.items()):
    ax.imshow(img)
    ax.set_title(ref_id, fontsize=11)
    ax.axis("off")
plt.tight_layout()
plt.show()

# 조합 3 검증

새 참조(ref-33~38)를 salon 사진에 적용해 눈이 깨지지 않는지 확인한다.
레포는 feat/face-restore — dev + 복원(⑥)·색 정합(⑦) 연결. PR ④ 검증을 겸한다.

In [ ]:
%cd /content
import os
import shutil
import sys

REPO = "/content/SalonCutAI"
shutil.rmtree(REPO, ignore_errors=True)
!git clone -q -b feat/face-restore https://github.com/qja0707/SalonCutAI.git {REPO}
!pip install -q diffusers==0.39.0 transformers==5.14.1 peft==0.19.1 accelerate==1.14.0 \
    insightface onnxruntime mediapipe==1.0.0 opencv-contrib-python-headless \
    facexlib torchvision

os.environ["IMAGE_GEN_ENABLED"] = "1"
os.environ["SALON_STORAGE_DIR"] = "/content/storage"
sys.path.insert(0, f"{REPO}/backend")
import importlib
importlib.invalidate_caches()

from src.ai_engine.image_gen import settings

!cd {REPO} && git branch --show-current
print("RESTORE_FIDELITY:", settings.RESTORE_FIDELITY)
print("IP_ADAPTER_SCALE:", settings.COMBO3_IP_ADAPTER_SCALE)

In [ ]:
import importlib
import shutil
from pathlib import Path

from src.ai_engine.image_gen import downloads, loader

# 새 참조를 레포 asset 에 넣어 실제 코드 경로(storage.ref_face_path)를 그대로 탄다
REF_SRC = Path("/content/drive/MyDrive/saloncut_data/ref_faces")
for i in range(33, 39):
    shutil.copy(REF_SRC / f"ref-{i}.png", settings.REF_FACES_DIR / f"ref-{i}.png")
print("asset/ref_faces:", len(list(settings.REF_FACES_DIR.glob("ref-*.png"))), "장")

downloads.ensure_models()
print("missing:", downloads.missing_files())

# warmup() 은 조합 5 까지 올려 VRAM 을 먹으므로 필요한 것만 올린다
loader.get_face_app()
loader.get_landmarker()
loader.get_segmenter()
loader.get_codeformer()
loader.get_face_helper()
loader.get_combo3()

import torch
print(f"VRAM {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
import time
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
from PIL import Image

from src.ai_engine.image_gen import pipeline, storage

SALON = Path("/content/drive/MyDrive/saloncut_data/test_images/salon")
OUT = Path("/content/drive/MyDrive/saloncut_data/outputs/ref-old_restore_0825")
OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
TARGETS = ["salon_02_long_wave_dark", "salon_04_long_wave_black"]
REFS = ["ref-01", "ref-02", "ref-09"]


def opts(ref_id):
    return SimpleNamespace(
        face=SimpleNamespace(mode="reference", reference=SimpleNamespace(reference_face_id=ref_id))
    )


for name in TARGETS:
    src = storage.to_stored_size(Image.open(SALON / f"{name}.jpg").convert("RGB"))
    fig, axes = plt.subplots(1, len(REFS) + 1, figsize=(4 * (len(REFS) + 1), 5))
    axes[0].imshow(src)
    axes[0].set_title("src", fontsize=11)
    axes[0].axis("off")

    for ax, ref_id in zip(axes[1:], REFS):
        t = time.time()
        fin = pipeline._run_reference_mode(src, opts(ref_id), SEED)
        fin.save(OUT / f"c3_{name}_{ref_id}.png")
        ax.imshow(fin)
        ax.set_title(ref_id, fontsize=11)
        ax.axis("off")
        print(f"{name}  {ref_id}  {time.time() - t:.0f}s")

    plt.tight_layout()
    plt.savefig(OUT / f"grid_{name}.png", dpi=100, bbox_inches="tight")
    plt.show()

- ref-39-44  8/25 추가. ref-33-38 이 비율 극단으로 이질감이 생겨,
             조화·인상 표현으로 바꿔 12장 후보 중 6장 채택.

In [ ]:
import base64
import io

BEAUTY3 = (
    "a Korean woman in her early 20s, breathtakingly beautiful, "
    "K-pop idol visual center level beauty, the kind of face that stops people "
    "in the street, perfect facial harmony and golden-ratio proportions, "
    "flawless luminous skin, captivating eyes"
)

FACES3 = {
    "A": "slim oval face with clean jawline, large bright double-lidded eyes, high straight nose bridge, soft full lips, classic elegant beauty",
    "B": "defined V-line face with gentle cheekbones, deep-set expressive eyes with long lashes, sculpted nose, full lips, glamorous impression",
    "C": "small round face with soft chin, large clear innocent eyes, small refined nose, pale luminous skin, pure youthful beauty",
    "D": "slim face with refined bone structure, elegant upturned almond eyes, thin high nose bridge, defined lips, sophisticated cat-like charm",
    "E": "small heart-shaped face, big round sparkling eyes, petite straight nose, small rosy lips, doll-like lovely beauty",
    "F": "elongated elegant face with strong clean jawline, long narrow eyes with cool confident gaze, high thin nose bridge, defined lips, chic aloof beauty",
}

CAND_DIR = REF_DIR / "candidates_0825"
CAND_DIR.mkdir(exist_ok=True)

cands = {}
for key, detail in FACES3.items():
    for n in (1, 2):
        prompt = f"{BEAUTY3}, {detail}, {SUFFIX}"
        t = time.time()
        res = client.images.generate(model="gpt-image-2", prompt=prompt, size="1024x1024", n=1)
        img = Image.open(io.BytesIO(base64.b64decode(res.data[0].b64_json))).convert("RGB")
        img.save(CAND_DIR / f"{key}{n}.png")
        cands[f"{key}{n}"] = img
        print(f"{key}{n}  {time.time() - t:.0f}s")

fig, axes = plt.subplots(2, 6, figsize=(24, 8))
for ax, (k, img) in zip(axes.flat, cands.items()):
    ax.imshow(img)
    ax.set_title(k, fontsize=11)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
from src.ai_engine.image_gen import combo3, compose, masks

OUT9 = Path("/content/drive/MyDrive/saloncut_data/outputs/cand_0825")
OUT9.mkdir(parents=True, exist_ok=True)

CANDS = [f"{k}{n}" for k in "ABCDEF" for n in (1, 2)]


def run_with_ref(src, ref_path, seed):
    """_run_reference_mode 와 같은 흐름. 참조 경로만 직접 받는다."""
    out, img_r, _ = combo3.generate(src, ref_path, seed)
    face_mask = masks.build_face_mask(img_r)
    hair_mask = masks.build_hair_mask(img_r)
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)
    out = compose.color_transfer(out, img_r, gen_mask)
    comp = compose.recompose_with_hair(img_r, out, face_mask, hair_mask)
    comp = compose.keep_brows(img_r, comp)
    return pipeline._restore_and_match(img_r, comp, gen_mask)


for name in TARGETS:
    src = storage.to_stored_size(Image.open(SALON / f"{name}.jpg").convert("RGB"))
    fig, axes = plt.subplots(2, len(CANDS), figsize=(3.2 * len(CANDS), 9))

    for col, cand in enumerate(CANDS):
        ref_path = CAND_DIR / f"{cand}.png"
        t = time.time()
        fin = run_with_ref(src, ref_path, SEED)
        fin.save(OUT9 / f"c3_{name}_{cand}.png")

        axes[0, col].imshow(Image.open(ref_path))
        axes[0, col].set_title(cand, fontsize=11)
        axes[0, col].axis("off")
        axes[1, col].imshow(fin)
        axes[1, col].axis("off")
        print(f"{name}  {cand}  {time.time() - t:.0f}s")

    plt.tight_layout()
    plt.savefig(OUT9 / f"grid_{name}.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
for cand in CANDS:
    fig, axes = plt.subplots(1, 3, figsize=(18, 7))
    axes[0].imshow(Image.open(CAND_DIR / f"{cand}.png"))
    axes[0].set_title(f"{cand} ref", fontsize=13)
    for ax, name in zip(axes[1:], TARGETS):
        ax.imshow(Image.open(OUT9 / f"c3_{name}_{cand}.png"))
        ax.set_title(name[:8], fontsize=13)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
srcs = {
    name: storage.to_stored_size(Image.open(SALON / f"{name}.jpg").convert("RGB"))
    for name in TARGETS
}

for cand in CANDS:
    fig, axes = plt.subplots(1, 5, figsize=(26, 7))
    axes[0].imshow(Image.open(CAND_DIR / f"{cand}.png"))
    axes[0].set_title(f"{cand} ref", fontsize=13)
    col = 1
    for name in TARGETS:
        axes[col].imshow(srcs[name])
        axes[col].set_title(f"{name[:8]} src", fontsize=13)
        axes[col + 1].imshow(Image.open(OUT9 / f"c3_{name}_{cand}.png"))
        axes[col + 1].set_title(f"{name[:8]} {cand}", fontsize=13)
        col += 2
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np

name = "salon_04_long_wave_black"
src = srcs[name]
det = loader.get_face_app().get(np.array(src)[:, :, ::-1])
x1, y1, x2, y2 = det[0].bbox.astype(int)
fw = x2 - x1
box = (x1 - fw // 4, y1 - fw // 4, x2 + fw // 4, y1 + fw // 2)

fig, axes = plt.subplots(1, 4, figsize=(20, 6))
axes[0].imshow(src.crop(box))
axes[0].set_title("src", fontsize=13)
for ax, cand in zip(axes[1:], ["A1", "C2", "F1"]):
    fin = Image.open(OUT9 / f"c3_{name}_{cand}.png").resize(src.size)
    ax.imshow(fin.crop(box))
    ax.set_title(cand, fontsize=13)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

fw_r = fw * 1024 / max(src.size)
print(f"img_r 얼굴 폭 약 {fw_r:.0f}px → 20px 는 {20 / fw_r:.1%}, 비율 0.077 이면 {max(1, int(fw_r * 0.077))}px")

In [ ]:
def run_with_ref_dilate(src, ref_path, seed, dilate):
    out, img_r, _ = combo3.generate(src, ref_path, seed)
    det = loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
    face_mask = masks.build_face_mask(img_r)
    hair_mask = masks.build_hair_mask(img_r, dilate=dilate)
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)
    out = compose.color_transfer(out, img_r, gen_mask)
    comp = compose.recompose_with_hair(img_r, out, face_mask, hair_mask)
    comp = compose.keep_brows(img_r, comp)
    return pipeline._restore_and_match(img_r, comp, gen_mask)


cand = "A1"
fin20 = Image.open(OUT9 / f"c3_{name}_{cand}.png").resize(src.size)
fin13 = run_with_ref_dilate(src, CAND_DIR / f"{cand}.png", SEED, dilate=13).resize(src.size)
fin13.save(OUT9 / f"c3_{name}_{cand}_dilate13.png")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (lab, im) in zip(axes, (("src", src), ("dilate 20", fin20), ("dilate 13", fin13))):
    ax.imshow(im.crop(box))
    ax.set_title(lab, fontsize=13)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
name = "salon_04_long_wave_black"
src = srcs[name]
cand = "A1"

det = loader.get_face_app().get(np.array(src)[:, :, ::-1])
x1, y1, x2, y2 = det[0].bbox.astype(int)
fw = x2 - x1
box = (x1 - fw // 4, y1 - fw // 4, x2 + fw // 4, y1 + fw // 2)

results = {"src": src, "seed 42": Image.open(OUT9 / f"c3_{name}_{cand}.png").resize(src.size)}
for seed in (7, 1234):
    fin = run_with_ref(src, CAND_DIR / f"{cand}.png", seed).resize(src.size)
    fin.save(OUT9 / f"c3_{name}_{cand}_seed{seed}.png")
    results[f"seed {seed}"] = fin

fig, axes = plt.subplots(2, 4, figsize=(24, 12))
for col, (lab, im) in enumerate(results.items()):
    axes[0, col].imshow(im)
    axes[0, col].set_title(lab, fontsize=13)
    axes[1, col].imshow(im.crop(box))
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
import random

from src.ai_engine.image_gen import combo3, compose, masks

OUT9 = Path("/content/drive/MyDrive/saloncut_data/outputs/cand_0825_rand")
OUT9.mkdir(parents=True, exist_ok=True)

CANDS = [f"{k}{n}" for k in "ABCDEF" for n in (1, 2)]
TARGETS = [
    "salon_01_long_wave_brown",
    "salon_02_long_wave_dark",
    "salon_03_long_wave_ring",
    "salon_04_long_wave_black",
    "salon_05_short_bob_brown",
]


def run_with_ref(src, ref_path, seed):
    """_run_reference_mode 와 같은 흐름. 참조 경로만 직접 받는다."""
    out, img_r, _ = combo3.generate(src, ref_path, seed)
    face_mask = masks.build_face_mask(img_r)
    hair_mask = masks.build_hair_mask(img_r)
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)
    out = compose.color_transfer(out, img_r, gen_mask)
    comp = compose.recompose_with_hair(img_r, out, face_mask, hair_mask)
    comp = compose.keep_brows(img_r, comp)
    return pipeline._restore_and_match(img_r, comp, gen_mask)


srcs = {
    name: storage.to_stored_size(Image.open(SALON / f"{name}.jpg").convert("RGB"))
    for name in TARGETS
}
seeds = {}

for name in TARGETS:
    src = srcs[name]
    fig, axes = plt.subplots(2, len(CANDS), figsize=(3.2 * len(CANDS), 9))

    for col, cand in enumerate(CANDS):
        ref_path = CAND_DIR / f"{cand}.png"
        seed = random.randint(0, 2**31 - 1)  # 서비스와 같은 방식
        t = time.time()
        fin = run_with_ref(src, ref_path, seed)
        fin.save(OUT9 / f"c3_{name}_{cand}_s{seed}.png")
        seeds[(name, cand)] = seed
        print(f"{name}  {cand}  seed {seed}  {time.time() - t:.0f}s")

        axes[0, col].imshow(Image.open(ref_path))
        axes[0, col].set_title(cand, fontsize=11)
        axes[0, col].axis("off")
        axes[1, col].imshow(fin)
        axes[1, col].set_title(f"s{seed}", fontsize=8)
        axes[1, col].axis("off")

    plt.tight_layout()
    plt.savefig(OUT9 / f"grid_{name}.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
import glob


def find_out(name, cand):
    hits = glob.glob(str(OUT9 / f"c3_{name}_{cand}_s*.png"))
    return Image.open(hits[0]) if hits else None


for cand in CANDS:
    fig, axes = plt.subplots(1, len(TARGETS) + 1, figsize=(5 * (len(TARGETS) + 1), 7))
    axes[0].imshow(Image.open(CAND_DIR / f"{cand}.png"))
    axes[0].set_title(f"{cand} ref", fontsize=13)
    for ax, name in zip(axes[1:], TARGETS):
        ax.imshow(find_out(name, cand))
        ax.set_title(f"{name[6:8]} {cand}", fontsize=13)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
REDO = [
    ("salon_02_long_wave_dark", "B2"),
    ("salon_03_long_wave_ring", "C1"),
    ("salon_03_long_wave_ring", "F1"),
]

for name, cand in REDO:
    src = srcs[name]
    ref_path = CAND_DIR / f"{cand}.png"
    row = {"src": src, "이전": find_out(name, cand)}

    for _ in range(2):
        seed = random.randint(0, 2**31 - 1)
        t = time.time()
        fin = run_with_ref(src, ref_path, seed)
        fin.save(OUT9 / f"c3_{name}_{cand}_s{seed}.png")
        row[f"s{seed}"] = fin
        print(f"{name}  {cand}  seed {seed}  {time.time() - t:.0f}s")

    fig, axes = plt.subplots(1, 4, figsize=(22, 7))
    for ax, (lab, im) in zip(axes, row.items()):
        ax.imshow(im)
        ax.set_title(f"{name[6:8]} {cand} {lab}", fontsize=13)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
FIG = Path("/content/drive/MyDrive/saloncut_data/outputs/report_figures")
FIG.mkdir(parents=True, exist_ok=True)


def save_grid(paths, titles, ncols, out, w=3.2):
    nrows = -(-len(paths) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(w * ncols, w * 1.15 * nrows))
    for ax, p, t in zip(axes.flat, paths, titles):
        ax.imshow(Image.open(p))
        ax.set_title(t, fontsize=11)
    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(out, dpi=100, bbox_inches="tight")
    plt.close(fig)
    print("saved", out.name)


# 3절 — 2차 참조 6장
ids = [f"ref-{i}" for i in range(33, 39)]
save_grid([REF_DIR / f"{i}.png" for i in ids], ids, 3, FIG / "fig106_ref33-38_extreme.png")

# 4절 — 3차 후보 12장
save_grid([CAND_DIR / f"{c}.png" for c in CANDS], CANDS, 6, FIG / "fig107_cand_A1-F2.png")

# 5절 — salon 5장 그리드를 세로로 합침
grids = [Image.open(OUT9 / f"grid_{n}.png") for n in TARGETS]
wmax = max(g.width for g in grids)
canvas = Image.new("RGB", (wmax, sum(g.height for g in grids)), "white")
y = 0
for g in grids:
    canvas.paste(g, (0, y))
    y += g.height
canvas.save(FIG / "fig108_cand_salon_all.png")
print("saved fig108_cand_salon_all.png", canvas.size)

In [ ]:
name = "salon_02_long_wave_dark"
OLD = Path("/content/drive/MyDrive/saloncut_data/outputs/ref-old_restore_0825")
NEW = Path("/content/drive/MyDrive/saloncut_data/outputs/cand_0825")

items = [("src", srcs[name])]
items += [(r, Image.open(OLD / f"c3_{name}_{r}.png")) for r in ["ref-01", "ref-02", "ref-09"]]
items += [(c, Image.open(NEW / f"c3_{name}_{c}.png")) for c in ["A1", "D1", "F2"]]

fig, axes = plt.subplots(1, 7, figsize=(28, 6))
for ax, (lab, im) in zip(axes, items):
    ax.imshow(im)
    ax.set_title(lab, fontsize=13)
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT9 / "cmp_old_new_02.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
name, cand = "salon_02_long_wave_dark", "B2"
hits = sorted(glob.glob(str(OUT9 / f"c3_{name}_{cand}_s*.png")))

items = [("src", srcs[name])] + [
    (Path(h).stem.split("_")[-1], Image.open(h)) for h in hits
]
fig, axes = plt.subplots(1, len(items), figsize=(6 * len(items), 7))
for ax, (lab, im) in zip(axes, items):
    ax.imshow(im)
    ax.set_title(f"{cand} {lab}", fontsize=13)
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT9 / f"seedcmp_02_{cand}.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
SWEEP = Path("/content/drive/MyDrive/saloncut_data/outputs/sweep_0825")
SWEEP.mkdir(parents=True, exist_ok=True)

STRENGTHS = [0.4, 0.5, 0.6]
IP_SCALES = [0.5, 0.65, 0.8]
REF = "D1"
pipe = loader.get_combo3()

for name in ["salon_02_long_wave_dark", "salon_04_long_wave_black"]:
    src = srcs[name]
    fig, axes = plt.subplots(len(STRENGTHS), len(IP_SCALES) + 1, figsize=(4.5 * (len(IP_SCALES) + 1), 5.5 * len(STRENGTHS)))

    for i, st in enumerate(STRENGTHS):
        axes[i, 0].imshow(src)
        axes[i, 0].set_title("src", fontsize=11)
        for j, ip in enumerate(IP_SCALES):
            settings.COMBO3_STRENGTH = st
            pipe.set_ip_adapter_scale(ip)
            t = time.time()
            fin = run_with_ref(src, CAND_DIR / f"{REF}.png", SEED)
            fin.save(SWEEP / f"c3_{name}_{REF}_st{st}_ip{ip}.png")
            axes[i, j + 1].imshow(fin)
            axes[i, j + 1].set_title(f"st {st}  ip {ip}", fontsize=11)
            print(f"{name}  st {st}  ip {ip}  {time.time() - t:.0f}s")

    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(SWEEP / f"grid_{name}_{REF}.png", dpi=100, bbox_inches="tight")
    plt.show()

# 원래 값으로 복구
settings.COMBO3_STRENGTH = 0.4
pipe.set_ip_adapter_scale(0.5)

In [ ]:
SWEEP = Path("/content/drive/MyDrive/saloncut_data/outputs/sweep_0825")
SWEEP.mkdir(parents=True, exist_ok=True)

STRENGTHS = [0.4, 0.6, 0.8]
IP_SCALES = [0.5, 0.8, 1.0]
REF = "D1"
pipe = loader.get_combo3()

for name in ["salon_01_long_wave_brown", "salon_02_long_wave_dark", "salon_04_long_wave_black"]:
    src = srcs[name]
    fig, axes = plt.subplots(len(STRENGTHS), len(IP_SCALES) + 1, figsize=(4.5 * (len(IP_SCALES) + 1), 5.5 * len(STRENGTHS)))

    for i, st in enumerate(STRENGTHS):
        axes[i, 0].imshow(src)
        axes[i, 0].set_title("src", fontsize=11)
        for j, ip in enumerate(IP_SCALES):
            settings.COMBO3_STRENGTH = st
            pipe.set_ip_adapter_scale(ip)
            t = time.time()
            fin = run_with_ref(src, CAND_DIR / f"{REF}.png", SEED)
            fin.save(SWEEP / f"c3_{name}_{REF}_st{st}_ip{ip}.png")
            axes[i, j + 1].imshow(fin)
            axes[i, j + 1].set_title(f"st {st}  ip {ip}", fontsize=11)
            print(f"{name}  st {st}  ip {ip}  {time.time() - t:.0f}s")

    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(SWEEP / f"grid_{name}_{REF}_wide.png", dpi=100, bbox_inches="tight")
    plt.show()

settings.COMBO3_STRENGTH = 0.4
pipe.set_ip_adapter_scale(0.5)

In [ ]:
REF = "B2"
CONFIGS = [(0.4, 0.5), (0.4, 1.0)]
pipe = loader.get_combo3()

fig, axes = plt.subplots(len(TARGETS) + 1, len(CONFIGS) + 1, figsize=(6 * (len(CONFIGS) + 1), 7 * (len(TARGETS) + 1)))

# 첫 행: 참조 얼굴
axes[0, 0].imshow(Image.open(CAND_DIR / f"{REF}.png"))
axes[0, 0].set_title(f"ref {REF}", fontsize=13)
axes[0, 1].axis("off")
axes[0, 2].axis("off")

for i, name in enumerate(TARGETS, start=1):
    src = srcs[name]
    axes[i, 0].imshow(src)
    axes[i, 0].set_title(f"{name[6:8]} src", fontsize=13)

    for j, (st, ip) in enumerate(CONFIGS):
        path = SWEEP / f"c3_{name}_{REF}_st{st}_ip{ip}.png"
        if path.exists():
            fin = Image.open(path)
        else:
            settings.COMBO3_STRENGTH = st
            pipe.set_ip_adapter_scale(ip)
            t = time.time()
            fin = run_with_ref(src, CAND_DIR / f"{REF}.png", SEED)
            fin.save(path)
            print(f"{name}  st {st}  ip {ip}  {time.time() - t:.0f}s")
        axes[i, j + 1].imshow(fin)
        axes[i, j + 1].set_title(f"{name[6:8]}  st {st}  ip {ip}", fontsize=13)

for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.savefig(SWEEP / f"cmp_ip05_vs_ip10_{REF}.png", dpi=100, bbox_inches="tight")
plt.show()

settings.COMBO3_STRENGTH = 0.4
pipe.set_ip_adapter_scale(0.5)

In [ ]:
REF = "B2"
CONFIGS = [(0.6, 0.5), (0.6, 1.0)]
pipe = loader.get_combo3()

fig, axes = plt.subplots(len(TARGETS) + 1, len(CONFIGS) + 1, figsize=(6 * (len(CONFIGS) + 1), 7 * (len(TARGETS) + 1)))

axes[0, 0].imshow(Image.open(CAND_DIR / f"{REF}.png"))
axes[0, 0].set_title(f"ref {REF}", fontsize=13)
axes[0, 1].axis("off")
axes[0, 2].axis("off")

for i, name in enumerate(TARGETS, start=1):
    src = srcs[name]
    axes[i, 0].imshow(src)
    axes[i, 0].set_title(f"{name[6:8]} src", fontsize=13)

    for j, (st, ip) in enumerate(CONFIGS):
        path = SWEEP / f"c3_{name}_{REF}_st{st}_ip{ip}.png"
        if path.exists():
            fin = Image.open(path)
        else:
            settings.COMBO3_STRENGTH = st
            pipe.set_ip_adapter_scale(ip)
            t = time.time()
            fin = run_with_ref(src, CAND_DIR / f"{REF}.png", SEED)
            fin.save(path)
            print(f"{name}  st {st}  ip {ip}  {time.time() - t:.0f}s")
        axes[i, j + 1].imshow(fin)
        axes[i, j + 1].set_title(f"{name[6:8]}  st {st}  ip {ip}", fontsize=13)

for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.savefig(SWEEP / f"cmp_st06_ip05_vs_ip10_{REF}.png", dpi=100, bbox_inches="tight")
plt.show()

settings.COMBO3_STRENGTH = 0.4
pipe.set_ip_adapter_scale(0.5)

In [ ]:
name, REF = "salon_04_long_wave_black", "B2"
st, ip = 0.6, 1.0
src = srcs[name]
pipe = loader.get_combo3()

settings.COMBO3_STRENGTH = st
pipe.set_ip_adapter_scale(ip)

row = {"src": src}
for seed in [SEED, 7, 1234]:
    path = SWEEP / f"c3_{name}_{REF}_st{st}_ip{ip}_s{seed}.png"
    t = time.time()
    fin = run_with_ref(src, CAND_DIR / f"{REF}.png", seed)
    fin.save(path)
    row[f"s{seed}"] = fin
    print(f"seed {seed}  {time.time() - t:.0f}s")

settings.COMBO3_STRENGTH = 0.4
pipe.set_ip_adapter_scale(0.5)

fig, axes = plt.subplots(1, 4, figsize=(24, 7))
for ax, (lab, im) in zip(axes, row.items()):
    ax.imshow(im)
    ax.set_title(f"04 {REF} st{st} ip{ip} {lab}", fontsize=12)
    ax.axis("off")
plt.tight_layout()
plt.savefig(SWEEP / f"seedcmp_04_{REF}_st{st}_ip{ip}.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
REF = "B2"
CN_SCALES = [0.8, 0.5, 0.3]
pipe = loader.get_combo3()
settings.COMBO3_STRENGTH = 0.4
pipe.set_ip_adapter_scale(0.5)

fig, axes = plt.subplots(len(TARGETS) + 1, len(CN_SCALES) + 1, figsize=(6 * (len(CN_SCALES) + 1), 7 * (len(TARGETS) + 1)))

axes[0, 0].imshow(Image.open(CAND_DIR / f"{REF}.png"))
axes[0, 0].set_title(f"ref {REF}", fontsize=13)
for ax in axes[0, 1:]:
    ax.axis("off")

for i, name in enumerate(TARGETS, start=1):
    src = srcs[name]
    axes[i, 0].imshow(src)
    axes[i, 0].set_title(f"{name[6:8]} src", fontsize=13)

    for j, cn in enumerate(CN_SCALES):
        path = SWEEP / f"c3_{name}_{REF}_st0.4_ip0.5_cn{cn}.png"
        if cn == 0.8 and (SWEEP / f"c3_{name}_{REF}_st0.4_ip0.5.png").exists():
            fin = Image.open(SWEEP / f"c3_{name}_{REF}_st0.4_ip0.5.png")
        elif path.exists():
            fin = Image.open(path)
        else:
            settings.COMBO3_CONTROLNET_SCALE = cn
            t = time.time()
            fin = run_with_ref(src, CAND_DIR / f"{REF}.png", SEED)
            fin.save(path)
            print(f"{name}  cn {cn}  {time.time() - t:.0f}s")
        axes[i, j + 1].imshow(fin)
        axes[i, j + 1].set_title(f"{name[6:8]}  cn {cn}", fontsize=13)

settings.COMBO3_CONTROLNET_SCALE = 0.8

for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.savefig(SWEEP / f"cmp_controlnet_{REF}.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
def pixdiff(a, b):
    x = np.asarray(Image.open(a), dtype=np.float32)
    y = np.asarray(Image.open(b), dtype=np.float32)
    return np.abs(x - y).mean()


name = "salon_02_long_wave_dark"
base = SWEEP / f"c3_{name}_B2_st0.4_ip0.5.png"
pairs = {
    "ip 0.5 vs 1.0 (st0.4)": (base, SWEEP / f"c3_{name}_B2_st0.4_ip1.0.png"),
    "ip 0.5 vs 1.0 (st0.6)": (SWEEP / f"c3_{name}_B2_st0.6_ip0.5.png", SWEEP / f"c3_{name}_B2_st0.6_ip1.0.png"),
    "cn 0.8 vs 0.5": (base, SWEEP / f"c3_{name}_B2_st0.4_ip0.5_cn0.5.png"),
    "cn 0.8 vs 0.3": (base, SWEEP / f"c3_{name}_B2_st0.4_ip0.5_cn0.3.png"),
    "st 0.4 vs 0.6 (ip0.5)": (base, SWEEP / f"c3_{name}_B2_st0.6_ip0.5.png"),
}
for k, (a, b) in pairs.items():
    print(f"{k:<26} 평균 픽셀 차이 {pixdiff(a, b):.2f}")

# ip 스케일이 실제로 attention processor 에 박혔는지
pipe = loader.get_combo3()
pipe.set_ip_adapter_scale(1.0)
scales = {getattr(p, "scale", None) for p in pipe.unet.attn_processors.values()}
print("set 1.0 후 processor scale 값들:", scales)
pipe.set_ip_adapter_scale(0.5)

In [ ]:
SUFFIX_SMILE = SUFFIX.replace(
    "neutral relaxed expression", "gentle natural closed-lip smile, warm expression"
)
prompt = f"{BEAUTY3}, {FACES3['B']}, {SUFFIX_SMILE}"

res = client.images.generate(model="gpt-image-2", prompt=prompt, size="1024x1024", n=1)
smile = Image.open(io.BytesIO(base64.b64decode(res.data[0].b64_json))).convert("RGB")
smile.save(CAND_DIR / "B2_smile.png")
print("참조 생성 완료")

REF = "B2_smile"
pipe = loader.get_combo3()
settings.COMBO3_STRENGTH = 0.4
pipe.set_ip_adapter_scale(0.5)
settings.COMBO3_CONTROLNET_SCALE = 0.8

fig, axes = plt.subplots(len(TARGETS) + 1, 3, figsize=(18, 7 * (len(TARGETS) + 1)))
axes[0, 0].imshow(Image.open(CAND_DIR / "B2.png"))
axes[0, 0].set_title("ref B2", fontsize=13)
axes[0, 1].imshow(smile)
axes[0, 1].set_title("ref B2_smile", fontsize=13)
axes[0, 2].axis("off")

for i, name in enumerate(TARGETS, start=1):
    src = srcs[name]
    axes[i, 0].imshow(src)
    axes[i, 0].set_title(f"{name[6:8]} src", fontsize=13)
    axes[i, 1].imshow(Image.open(SWEEP / f"c3_{name}_B2_st0.4_ip0.5.png"))
    axes[i, 1].set_title(f"{name[6:8]} B2", fontsize=13)

    t = time.time()
    fin = run_with_ref(src, CAND_DIR / f"{REF}.png", SEED)
    fin.save(SWEEP / f"c3_{name}_{REF}_st0.4_ip0.5.png")
    axes[i, 2].imshow(fin)
    axes[i, 2].set_title(f"{name[6:8]} B2_smile", fontsize=13)
    print(f"{name}  {time.time() - t:.0f}s")

for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.savefig(SWEEP / "cmp_B2_vs_smile.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
REF = "B2"
st, ip = 0.6, 0.5
SEEDS = [42, 7, 1234, 2024, 99, 314, 777, 1001, 555, 8080]
pipe = loader.get_combo3()
settings.COMBO3_STRENGTH = st
pipe.set_ip_adapter_scale(ip)
settings.COMBO3_CONTROLNET_SCALE = 0.8

for name in ["salon_01_long_wave_brown", "salon_04_long_wave_black"]:
    src = srcs[name]
    fig, axes = plt.subplots(2, 6, figsize=(30, 14))
    axes[0, 0].imshow(src)
    axes[0, 0].set_title(f"{name[6:8]} src", fontsize=13)
    axes[1, 0].axis("off")

    for k, seed in enumerate(SEEDS):
        r, c = divmod(k, 5)
        path = SWEEP / f"c3_{name}_{REF}_st{st}_ip{ip}_s{seed}.png"
        if path.exists():
            fin = Image.open(path)
        else:
            t = time.time()
            fin = run_with_ref(src, CAND_DIR / f"{REF}.png", seed)
            fin.save(path)
            print(f"{name}  seed {seed}  {time.time() - t:.0f}s")
        axes[r, c + 1].imshow(fin)
        axes[r, c + 1].set_title(f"s{seed}", fontsize=12)

    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(SWEEP / f"seeds10_{name[6:8]}_{REF}_st{st}.png", dpi=100, bbox_inches="tight")
    plt.show()

settings.COMBO3_STRENGTH = 0.4
pipe.set_ip_adapter_scale(0.5)

In [ ]:
REF = "B2"
st, ip = 0.4, 0.5
SEEDS = [42, 7, 1234, 2024, 99, 314, 777, 1001, 555, 8080]
pipe = loader.get_combo3()
settings.COMBO3_STRENGTH = st
pipe.set_ip_adapter_scale(ip)
settings.COMBO3_CONTROLNET_SCALE = 0.8

for name in ["salon_01_long_wave_brown", "salon_04_long_wave_black"]:
    src = srcs[name]
    fig, axes = plt.subplots(2, 6, figsize=(30, 14))
    axes[0, 0].imshow(src)
    axes[0, 0].set_title(f"{name[6:8]} src", fontsize=13)
    axes[1, 0].axis("off")

    for k, seed in enumerate(SEEDS):
        r, c = divmod(k, 5)
        path = SWEEP / f"c3_{name}_{REF}_st{st}_ip{ip}_s{seed}.png"
        if path.exists():
            fin = Image.open(path)
        else:
            t = time.time()
            fin = run_with_ref(src, CAND_DIR / f"{REF}.png", seed)
            fin.save(path)
            print(f"{name}  seed {seed}  {time.time() - t:.0f}s")
        axes[r, c + 1].imshow(fin)
        axes[r, c + 1].set_title(f"s{seed}", fontsize=12)

    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(SWEEP / f"seeds10_{name[6:8]}_{REF}_st{st}.png", dpi=100, bbox_inches="tight")
    plt.show()